# E-commerce A/B Test Analysis — Exploration

## The business context

A mid-sized online retailer ran a 3-week A/B test of a redesigned landing page, expecting to improve **conversion rate** (% of visitors who complete a purchase). Before committing engineering resources to a full rollout, they need to answer one question:

> **Did the new landing page actually increase conversion rate, or were any differences just noise?**

## Who is this for?

- **Product team** — deciding whether to ship the new design
- **Engineering leadership** — deciding whether to commit rollout resources  
- **CMO** — deciding whether this design pattern should influence other site changes

## What's at stake?

With ~5M annual visitors and a baseline conversion rate of ~12%, a **1 percentage point lift** in conversion would generate roughly 2.5 million in additional annual revenue (assuming ~$50 average order value). The cost of a false positive — shipping a page that doesn't actually work — includes engineering resources, potential new bugs, and opportunity cost against other roadmap items.

## Goal of this notebook

Explore only. Understand the dataset, its structure, and whether the randomization and experiment setup look sound. No cleaning or analysis yet. 


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

In [4]:
df = pd.read_csv("/Users/edi/Documents/ab-test-analysis/data/ab_data.csv")

In [7]:
# see how many rows and columns we are working with
df.shape

(294478, 5)

In [8]:
# view the first 5 rows of the data 
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [9]:
# see how many null values we have in each column

df.isnull().sum()

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

Looks like there are no null values

## Understanding the columns

- **user_id** tells us the unique user identifier
- **timestamp** tells us when the user visited the landing page
- **group** tells us the experimental assignment: `control` (old page) or `treatment` (new page)
- **landing_page** tells us which page the user **actually saw**: `old_page` or `new_page`
- **converted** tells us two things. 1 if the user completed a purchase, 0 otherwise

**Critical distinction:** `group` is the assignment. `landing_page` is what actually happened. These should always match. But in implementation bugs can cause them to diverge. Finding and handling this mismatch is a major part of cleaning.

In [18]:
# how many users in each group?

group_counts = df.groupby("group").size()

print("Users per group:")
print(group_counts)

total = len(df)
print(f"\nTotal rows: {total:,}")

print(f"Control share: {group_counts['control']/total*100:.2f}%")
print(f"Treatment share: {group_counts['treatment']/total*100:.2f}%")

Users per group:
group
control      147202
treatment    147276
dtype: int64

Total rows: 294,478
Control share: 49.99%
Treatment share: 50.01%


In [20]:
# check to confirm that the group assignment matches the landing page shown
mismatch = df[
    ((df["group"] == "control") & (df["landing_page"] != "old_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] != "new_page"))
]
print(f"Mismatched rows: {len(mismatch):,} ({len(mismatch)/len(df)*100:.2f}% of data)")
print(f"\nThese are users who were assigned to one group but shown the wrong page.")
print(f"We can't include them in the analysis since we don't know what they're measuring.")
print(f"\nSample of mismatched rows:")
mismatch.head()

Mismatched rows: 3,893 (1.32% of data)

These are users who were assigned to one group but shown the wrong page.
We can't include them in the analysis since we don't know what they're measuring.

Sample of mismatched rows:


,user_id,timestamp,group,landing_page,converted
22,767017,2017-01-12 22:58:14.991443,control,new_page,0
240,733976,2017-01-11 15:11:16.407599,control,new_page,0
308,857184,2017-01-20 07:34:59.832626,treatment,old_page,0
327,686623,2017-01-09 14:26:40.734775,treatment,old_page,0
357,856078,2017-01-12 12:29:30.354835,treatment,old_page,0


In [22]:
# check to see if there there are users appearing more than once?
user_counts = df["user_id"].value_counts()
dupes = user_counts[user_counts >1]

print(f"Users appearing more than once: {len(dupes):,}")
print(f"Total duplicate-appearance rows: {dupes.sum() - len(dupes):,}")
print(f"\nSample of duplicated users:")
df[df["user_id"].isin(dupes.head(5).index)].sort_values("user_id")

Users appearing more than once: 3,894
Total duplicate-appearance rows: 3,894

Sample of duplicated users:


,user_id,timestamp,group,landing_page,converted
105487,722274,2017-01-19 01:46:53.093257,control,old_page,0
262554,722274,2017-01-09 21:21:23.638444,control,new_page,0
236842,754884,2017-01-18 20:10:48.538515,treatment,old_page,0
245524,754884,2017-01-17 23:46:45.624036,treatment,new_page,0
22271,783176,2017-01-13 18:21:20.193262,treatment,new_page,0
35716,783176,2017-01-10 02:41:30.859607,control,new_page,0
90794,805339,2017-01-12 08:45:15.557052,treatment,old_page,0
234436,805339,2017-01-22 23:00:09.005288,control,old_page,0
264566,898232,2017-01-16 00:10:10.069015,control,new_page,0
287495,898232,2017-01-09 09:32:42.057795,treatment,new_page,0


In [28]:
# view conversion rates per group before cleaning

conv_by_group = df.groupby("group")["converted"].agg(["count", "sum", "mean"])

conv_by_group.columns = ["n_users", "n_converted", "conversion_rate"]

conv_by_group["conversion_pct"] = (conv_by_group["conversion_rate"] * 100).round(3)
print("Raw conversion rates by group (BEFORE cleaning):")
conv_by_group

Raw conversion rates by group (BEFORE cleaning):


,n_users,n_converted,conversion_rate,conversion_pct
group,,,,
control,147202,17723,0.1204,12.0400
treatment,147276,17514,0.1189,11.8920


In [29]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Experiment start: {df['timestamp'].min()}")
print(f"Experiment end:   {df['timestamp'].max()}")
print(f"Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")

Experiment start: 2017-01-02 13:42:05.378582
Experiment end:   2017-01-24 13:41:54.460509
Duration: 21 days
